# 04-O3 — Build master dataset & candidate feature pool (Medellín, O3 target)

Mirrors Bogotá's `05_o3_build_master_dataset.ipynb`. `o3` in
`medellin_network_daily_clean.parquet` is **already MDA8** (notebook 01 computes it
directly from hourly data, same method as Bogotá's notebook 04c) -- no separate
target-correction step needed here, unlike Bogotá where the SISAIRE daily mean had to be
replaced after the fact.

**Coverage note**: O3 and PM10 (needed for `pmc`) only start in 2019 in the Medellín
`Hourmean` source (checked in notebook 01) -- this analysis is therefore limited to
2019-2022, shorter than Medellín's own PM2.5-only analysis (2015-2022).


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from common.external_data import FIRE_SOURCE_REGIONS, bearing_from_bogota, MEDELLIN_LAT, MEDELLIN_LON

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "medellin"
FIG_DIR = PROJECT_ROOT / "Scripts" / "colombia" / "medellin" / "figures"

network = pd.read_parquet(DATA_DIR / "medellin_network_daily_clean.parquet")
era5 = pd.read_parquet(DATA_DIR / "medellin_era5_daily.parquet")
hotspots = pd.read_parquet(DATA_DIR / "medellin_hotspots_daily.parquet")
w700 = pd.read_parquet(DATA_DIR / "medellin_w700_daily.parquet")
antioquia = pd.read_parquet(DATA_DIR / "medellin_antioquia_network_daily.parquet")
antioquia_new = pd.read_parquet(DATA_DIR / "medellin_antioquia_network_new_daily.parquet")  # CO/SO2/etc. batch, 2026-09-23
print("antioquia_new (CO/SO2/etc.):", antioquia_new.shape, list(antioquia_new.columns))
lts = pd.read_parquet(DATA_DIR / "medellin_lts_daily.parquet")

# TROPOMI HCHO -- real satellite-derived VOC proxy. Medellin's O3 record already
# starts in 2019 (post-HCHO-launch), so effective coverage clears the pipeline's
# >=70% threshold (unlike Bogota, whose longer 2014-2022 O3 record does not).
hcho = pd.read_csv(PROJECT_ROOT / "Data" / "external" / "tropomi_hcho" / "medellin_hcho_daily.csv", parse_dates=["date"])
hcho = hcho.rename(columns={"hcho_trop_col_mol_m2": "hcho"})[["date", "hcho"]]
station_meteo_check = pd.read_parquet(DATA_DIR / "medellin_meteo_daily_clean.parquet")
print("station meteo:", station_meteo_check.shape, station_meteo_check["date"].min(), station_meteo_check["date"].max())

print("network:", network.shape)
print("o3 (MDA8) non-null:", network['o3'].notna().sum())
print("pm10 non-null:", network['pm10'].notna().sum())

print("hcho:", hcho.shape, "non-null:", hcho['hcho'].notna().sum())

antioquia_new (CO/SO2/etc.): (5331, 9) ['date', 'co', 'so2', 'no_sisaire', 'nox_reported', 'pst', 'radiation_station', 'temp_10cm', 'precip_station']
station meteo: (2919, 5) 2013-01-01 00:00:00 2020-12-31 00:00:00
network: (3652, 6)
o3 (MDA8) non-null: 1423
pm10 non-null: 1426
hcho: (2802, 2) non-null: 1961


## Merge

Three-source cascade, primary-with-calibrated-fallback at each level (same technique as
the station-vs-ERA5 meteorology fix earlier -- linear calibration on the overlap period
before filling gaps, never a raw splice, to avoid another Simpson's-paradox-style
level-shift artifact):

1. **`pm25`/`pm10`/`no2`/`wind_speed`/`wind_dir`**: primary source is now the
   Aburra-Valley-filtered, properly QC'd Antioquia SISAIRE network (notebook 00 --
   Chauvenet + >=75% completeness, 43-55 stations, 2008-2025), calibrated-fallback is
   whatever the previous primary source was for that variable (`Hourmean` for the
   pollutants, the existing station+ERA5 hybrid for wind) to fill occasional gaps within
   the Antioquia record. This replaces a shorter (2015-2022), undocumented-provenance
   `Hourmean` network mean with a transparent, much longer, larger-network-averaged
   series for the pollutants, and extends wind past the previous 2020 cutoff.
2. **`temp_10m`/`rh_10m`**: new standalone columns, straight from the Antioquia network
   (no prior 10m-height source existed to blend with) -- `temp_2m`/`rh_2m` (from the
   station+ERA5 hybrid) are unchanged and kept alongside.
3. **`pressure` dropped from the Antioquia source entirely** -- most of that network's
   barometric sensors are miscalibrated (per-station means ranging 551-872 hPa for a
   city where physically-correct is ~800-870 hPa; only 4/52 stations were plausible).
   ERA5's `pressure` remains the sole source, as before.
4. `no`/`o3` are unaffected -- still from the `Hourmean`-sourced `network` table (O3
   stays MDA8, not rebuilt from Antioquia's plain daily-mean `O3` column, to avoid
   silently blending two different ozone metrics -- see notebook 00's markdown).

(This notebook's target, `o3`, is untouched by any of the above -- only the candidate
predictor columns pm25/pm10/no2/wind_speed/wind_dir/temp_10m/rh_10m are affected.)


In [2]:
station_meteo = pd.read_parquet(DATA_DIR / "medellin_meteo_daily_clean.parquet")
station_meteo = station_meteo.rename(columns={
    'temp_2m': 'temp_2m_station', 'rh_2m': 'rh_2m_station',
    'wind_speed': 'wind_speed_station', 'wind_dir': 'wind_dir_station',
})

antioquia_r = antioquia.rename(columns={
    'pm25': 'pm25_antioquia', 'pm10': 'pm10_antioquia', 'no2': 'no2_antioquia',
    'wind_speed': 'wind_speed_antioquia', 'wind_dir': 'wind_dir_antioquia',
})

df = (network.rename(columns={'pm25': 'pm25_hourmean', 'pm10': 'pm10_hourmean', 'no2': 'no2_hourmean'})
             .merge(era5, on='date', how='left')
             .merge(hotspots, on='date', how='left')
             .merge(station_meteo, on='date', how='left')
             .merge(w700, on='date', how='left')
             .merge(lts, on='date', how='left')
             .merge(antioquia_r, on='date', how='left')
             .merge(antioquia_new, on='date', how='left')  # CO/SO2/etc. batch, 2026-09-23
             .merge(hcho, on='date', how='left'))
df = df.sort_values('date').reset_index(drop=True)

df = df.rename(columns={
    'wind_speed': 'wind_speed_era5', 'wind_dir': 'wind_dir_era5',
    'temp_2m': 'temp_2m_era5', 'rh_2m': 'rh_2m_era5',
})

from sklearn.linear_model import LinearRegression

def calibrate_and_fill(data, primary_col, fallback_col, label):
    overlap = data[[primary_col, fallback_col]].dropna()
    lr = LinearRegression().fit(overlap[[fallback_col]], overlap[primary_col])
    r2 = lr.score(overlap[[fallback_col]], overlap[primary_col])
    print(f"{label}: calibration {fallback_col}->{primary_col} scale  slope={lr.coef_[0]:.3f}  "
          f"intercept={lr.intercept_:.3f}  overlap_R2={r2:.3f}  (n={len(overlap)})")
    fallback_calibrated = lr.predict(data[[fallback_col]].fillna(0))
    fallback_calibrated = pd.Series(fallback_calibrated, index=data.index).where(data[fallback_col].notna())
    return data[primary_col].combine_first(fallback_calibrated)

# Station meteo -> ERA5 calibration (unchanged from the earlier fix)
for var in ['wind_speed', 'temp_2m', 'rh_2m']:
    df[f'{var}_hybrid_station_era5'] = calibrate_and_fill(df, f'{var}_station', f'{var}_era5', var)
df['wind_dir_hybrid_station_era5'] = df['wind_dir_station'].combine_first(df['wind_dir_era5'])

# Antioquia (primary) -> previous-hybrid-or-Hourmean (fallback) calibration
df['pm25'] = calibrate_and_fill(df, 'pm25_antioquia', 'pm25_hourmean', 'pm25')
df['pm10'] = calibrate_and_fill(df, 'pm10_antioquia', 'pm10_hourmean', 'pm10')
df['no2'] = calibrate_and_fill(df, 'no2_antioquia', 'no2_hourmean', 'no2')
df['wind_speed'] = calibrate_and_fill(df, 'wind_speed_antioquia', 'wind_speed_hybrid_station_era5', 'wind_speed')
df['wind_dir'] = df['wind_dir_antioquia'].combine_first(df['wind_dir_hybrid_station_era5'])
df['temp_2m'] = df['temp_2m_hybrid_station_era5']
df['rh_2m'] = df['rh_2m_hybrid_station_era5']
# temp_10m/rh_10m come straight from antioquia_r, already correctly named -- no action needed.

print(df.shape)
for v in ['pm25', 'pm10', 'no2', 'wind_speed', 'wind_dir', 'temp_10m', 'rh_10m']:
    print(f"{v}: total non-null={df[v].notna().sum()}")


wind_speed: calibration wind_speed_era5->wind_speed_station scale  slope=0.843  intercept=0.781  overlap_R2=0.494  (n=2193)
temp_2m: calibration temp_2m_era5->temp_2m_station scale  slope=1.196  intercept=-1.446  overlap_R2=0.767  (n=2193)
rh_2m: calibration rh_2m_era5->rh_2m_station scale  slope=0.857  intercept=-3.812  overlap_R2=0.727  (n=2193)
pm25: calibration pm25_hourmean->pm25_antioquia scale  slope=0.761  intercept=2.336  overlap_R2=0.752  (n=1736)
pm10: calibration pm10_hourmean->pm10_antioquia scale  slope=0.868  intercept=6.739  overlap_R2=0.901  (n=1426)
no2: calibration no2_hourmean->no2_antioquia scale  slope=0.262  intercept=29.768  overlap_R2=0.077  (n=1771)
wind_speed: calibration wind_speed_hybrid_station_era5->wind_speed_antioquia scale  slope=0.460  intercept=0.654  overlap_R2=0.177  (n=2898)
(3652, 65)
pm25: total non-null=3510
pm10: total non-null=2445
no2: total non-null=3615
wind_speed: total non-null=3652
wind_dir: total non-null=3652
temp_10m: total non-null=

## Engineered features

Same as Bogotá's O3 pipeline: `pmc`, calendar terms, ventilation, wind-transport-
conditioned hotspots (re-centered on Medellín), stagnation terms, `photochemical_potential`
(NO2 x radiation), `radiation_effective` (aerosol-shading hypothesis), and `no_ratio`
(NO2/NO -- the titration-strength term that dominated Bogotá's discovered equation).


In [3]:
df['pmc'] = df['pm10'] - df['pm25']

df['wind_dir_sin'] = np.sin(np.deg2rad(df['wind_dir']))
df['wind_dir_cos'] = np.cos(np.deg2rad(df['wind_dir']))
doy = df['date'].dt.dayofyear
df['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)
month = df['date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * month / 12)
df['month_cos'] = np.cos(2 * np.pi * month / 12)
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

df['ventilation_station'] = df['blh_mean'] * df['wind_speed']

SECTOR_BEARINGS = {name: i * 45.0 for i, name in enumerate(["N", "NE", "E", "SE", "S", "SW", "W", "NW"])}
point_region_cols = [c for c in df.columns if c.startswith('hotspots_') and not c.startswith('hotspots_sector_')]
sector_cols = [c for c in df.columns if c.startswith('hotspots_sector_')]

for region_col in point_region_cols:
    region_name = region_col.replace('hotspots_', '')
    bearing = bearing_from_bogota(
        FIRE_SOURCE_REGIONS[region_name]['lat'], FIRE_SOURCE_REGIONS[region_name]['lon'],
        origin_lat=MEDELLIN_LAT, origin_lon=MEDELLIN_LON,
    )
    alignment = np.cos(np.deg2rad(df['wind_dir'] - bearing)).clip(lower=0)
    df[f'{region_col}_transport'] = df[region_col] * alignment
for sector_col in sector_cols:
    sector_name = sector_col.replace('hotspots_sector_', '')
    bearing = SECTOR_BEARINGS[sector_name]
    alignment = np.cos(np.deg2rad(df['wind_dir'] - bearing)).clip(lower=0)
    df[f'{sector_col}_transport'] = df[sector_col] * alignment

df['ventilation_min'] = df['blh_min'] * df['wind_speed']
df['fog_stagnation'] = df['rh_2m'] / (1 + df['wind_speed'])
df['dry_stagnation'] = 1 / ((1 + df['wind_speed']) * (1 + df['precip_total_mm']))

df['photochemical_potential'] = df['no2'] * df['radiation_mean']
df['radiation_effective'] = df['radiation_mean'] / (1 + df['pm25'])
df['no_ratio'] = np.where(df['no'] > 1e-3, df['no2'] / df['no'], np.nan)


In [4]:
LAG_DAYS = [1, 2, 3]
hotspot_cols = [c for c in df.columns if c.startswith('hotspots_')]

LAG_VARS = [
    'co', 'so2', 'no_sisaire', 'radiation_station', 'temp_10cm', 'precip_station',  # new Antioquia batch, 2026-09-23 (pst excluded: 0 qualifying station-years; nox_reported excluded: only 2013-2015)
        'pm25', 'pmc', 'no2', 'no', 'no_ratio', 'wind_speed', 'temp_2m', 'rh_2m', 'pressure',
    'blh_mean', 'blh_morning_mean', 'radiation_mean', 'precip_total_mm',
    'ventilation_station', 'ventilation_min',
    'fog_stagnation', 'dry_stagnation', 'photochemical_potential', 'radiation_effective',
    'tcwv', 'sensible_heat_flux', 'tke_proxy', 'w700', 'lts', 'lts_morning',
    'temp_10m', 'rh_10m', 'hcho',
] + hotspot_cols
df = df.set_index('date')
for var in LAG_VARS:
    if var not in df.columns:
        continue
    for lag in LAG_DAYS:
        df[f'{var}_lag{lag}'] = df[var].shift(lag)

df['o3_lag1'] = df['o3'].shift(1)
df = df.reset_index()

candidate_cols = [c for c in df.columns if c not in ('date', 'o3')]
print(f"{len(candidate_cols)} candidate predictor columns")


259 candidate predictor columns


/var/folders/30/gtfddbxx5z3f_hc8v4br4pkh0000gn/T/ipykernel_53345/362832145.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{var}_lag{lag}'] = df[var].shift(lag)
/var/folders/30/gtfddbxx5z3f_hc8v4br4pkh0000gn/T/ipykernel_53345/362832145.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{var}_lag{lag}'] = df[var].shift(lag)
/var/folders/30/gtfddbxx5z3f_hc8v4br4pkh0000gn/T/ipykernel_53345/362832145.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

## Verification: same mechanism check as Bogotá

`no_ratio`/`no2`/`photochemical_potential` should correlate positively with O3 (NOx +
sunlight, weaker titration); `no` negatively (titration).


In [5]:
for col in ['pm25', 'pmc', 'no2', 'no', 'no_ratio', 'radiation_mean', 'photochemical_potential', 'radiation_effective', 'wind_speed']:
    if col in df.columns:
        corr = df[['o3', col]].dropna().corr().iloc[0, 1]
        n = df[['o3', col]].dropna().shape[0]
        print(f"corr(o3, {col}) = {corr:.3f}  (n={n})")


corr(o3, pm25) = 0.301  (n=1421)
corr(o3, pmc) = -0.001  (n=1421)
corr(o3, no2) = 0.113  (n=1423)
corr(o3, no) = -0.207  (n=1404)
corr(o3, no_ratio) = 0.242  (n=1404)
corr(o3, radiation_mean) = 0.264  (n=1423)
corr(o3, photochemical_potential) = 0.294  (n=1423)
corr(o3, radiation_effective) = -0.033  (n=1421)
corr(o3, wind_speed) = 0.127  (n=1423)


## Save O3 master dataset

In [6]:
out_path = DATA_DIR / "medellin_o3_master_dataset.parquet"
df.to_parquet(out_path, index=False)
print("Saved:", out_path, df.shape)
print("Rows with non-null target (o3):", df['o3'].notna().sum())


Saved: /path/to/project/Data/processed/colombia/medellin/medellin_o3_master_dataset.parquet (3652, 261)
Rows with non-null target (o3): 1423
